# OTTO-Scraper: Datenqualität, Verlauf und Produkte

Dieses Notebook untersucht `otto_produkte.db` Schritt für Schritt. Wir lernen die Daten kennen, prüfen die täglichen Scrapes, suchen nach Lücken, vergleichen Produktbestände und betrachten die Preisverteilung.

**So arbeitest du damit:** Führe die Zellen von oben nach unten aus (`Shift` + `Enter`). Ändere später die Parameter in den entsprechend markierten Zellen und führe die darunterliegenden Zellen erneut aus. Das Notebook liest bei jedem vollständigen Durchlauf die aktuelle Datenbank.

**Start:** Im Projektordner zum Beispiel `uv run --with jupyter jupyter lab notebooks/otto_scraper_monitoring.ipynb`. Alternativ das Notebook in VS Code mit der Jupyter-Erweiterung öffnen. Die Visualisierungen verwenden Altair, das über die Projektabhängigkeit Streamlit installiert wird.

> Wichtig: Die Datenbank speichert Produktzeilen, aber kein separates Protokoll über gestartete Scraper-Läufe oder fehlgeschlagene Suchbegriffe. Ein leerer Tag ist deshalb ein möglicher Aussetzer, aber kein Beweis dafür. Diese Einschränkung greifen wir am Ende wieder auf.

## 1. Bibliotheken und Einstellungen

Wir verwenden **pandas** zum Auswerten der Tabellen und **Altair** für interaktive Diagramme (mit Tooltips beim Darüberfahren). `DB_PATH` zeigt standardmäßig auf die Datenbank im Projektordner. Falls du das Notebook von einem anderen Arbeitsverzeichnis startest, wird zusätzlich relativ zum Notebook gesucht.

In [1]:
from pathlib import Path
import sqlite3

import altair as alt
import pandas as pd
from IPython.display import display, Markdown

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)

CANDIDATE_PATHS = [
    Path.cwd() / 'otto_produkte.db',
    Path.cwd().parent / 'otto_produkte.db',
]
DB_PATH = next((p.resolve() for p in CANDIDATE_PATHS if p.exists()), CANDIDATE_PATHS[0].resolve())
DB_PATH

WindowsPath('C:/Users/JONASCHN/YAHOO/otto_produkte.db')

### Mitgestalten: Datenbankpfad

Wenn oben nicht der richtige Pfad angezeigt wird, ersetze `DB_PATH` zum Beispiel durch `Path(r'C:\mein\projekt\otto_produkte.db')`. Das `r` vor dem Text verhindert, dass Backslashes als Sonderzeichen interpretiert werden.

In [2]:
if not DB_PATH.exists():
    raise FileNotFoundError(f'Datenbank nicht gefunden: {DB_PATH}')

with sqlite3.connect(DB_PATH) as connection:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        connection,
    )
    if 'produkte' not in tables['name'].tolist():
        raise ValueError(f'In {DB_PATH.name} wurde keine Tabelle "produkte" gefunden.')
    produkte = pd.read_sql_query('SELECT * FROM produkte', connection)

if produkte.empty:
    raise ValueError('Die Tabelle produkte enthält keine Zeilen.')

print(f'Datenbank: {DB_PATH}')
print(f'Zeilen: {len(produkte):,} | Spalten: {len(produkte.columns)}')
display(tables)
display(produkte.head())

Datenbank: C:\Users\JONASCHN\YAHOO\otto_produkte.db
Zeilen: 7,129 | Spalten: 16


,name
0,produkte
1,sqlite_sequence


,id,scraped_date,suchbegriff,titel,marke,preis,old_price,currency,bild_url,produkt_url,bewertung,anzahl_bewertungen,verfuegbarkeit,sku,gtin,created_at
0,1,2026-09-18,200k Teppich,OTTO home Teppich Salsa,OTTO home,7.99,20.99,EUR,https://i.otto.de/i/otto/29d94821-3ef6-5580-8719-330accca9994?$formatz$,https://www.otto.de/p/otto-home-teppich-salsa-rechteckig-hoehe-9-mm-mit-besonders-weichem-flor-k...,4.5,845.0,"bis Mi., 23. Sept. bei dir",1668711404,,2026-09-18 08:10:13
1,2,2026-09-18,200k Teppich,"OTTO home Teppich Kylan, in Standard- und Premium-Qualität, 9 mm oder 14 mm Höhe",OTTO home,12.99,28.70,EUR,https://i.otto.de/i/otto/bb1b3862-35f2-5147-ba88-0395264596f7?$formatz$,https://www.otto.de/p/otto-home-teppich-kylan-in-standard-und-premium-qualitaet-9-mm-oder-14-mm-...,4.5,234.0,"bis Mi., 23. Sept. bei dir",1854442039,,2026-09-18 08:10:13
2,16,2026-09-18,1-Euro Bleistift,"lindahandel Kugelschreiber 2-in-1 Kugelschreiber mit Touchscreen Stift, blaue Mine",lindahandel,17.99,28.78,EUR,https://i.otto.de/i/otto/c492b27a-b8eb-41bb-a39d-1da4d739f7e7?$formatz$,https://www.otto.de/p/lindahandel-kugelschreiber-2-in-1-kugelschreiber-mit-touchscreen-stift-bla...,NaN,NaN,"bis Do., 24. Sept. bei dir",S0JJM0GM,,2026-09-18 08:10:19
3,42,2026-09-18,Goldrand Tasse,"OTTO home Geschirr-Set Tommik, außen matt, innen glänzend",OTTO home,49.99,119.99,EUR,https://i.otto.de/i/otto/abb2c3b9-30c9-5e63-9988-7ebd38d8e15b?$formatz$,https://www.otto.de/p/otto-home-geschirr-set-tommik-aussen-matt-innen-glaenzend-16-tlg-4-persone...,4.5,48.0,in 2-4 Werktagen bei dir,1956271584,,2026-09-18 08:10:28
4,87,2026-09-18,Wetterstation Pro,"Hama Wetterstation Wetterstation Funk mit Außensensor (digital, Vorhersage)",Hama,59.99,69.99,EUR,https://i.otto.de/i/otto/e6c17fb0-1ef1-5369-b0b5-9790c72766c8?$formatz$,https://www.otto.de/p/hama-wetterstation-funk-mit-aussensensor-digital-vorhersage-wetterstation-...,4.5,20.0,in 3-4 Werktagen bei dir,C2001032124,,2026-09-18 08:10:47


## 2. Schema und Datenqualität kennenlernen

Jede Zeile ist ein Produktfund an einem Scrape-Tag. Laut Scraper verhindert die Datenbank einen zweiten Eintrag derselben `produkt_url` am selben `scraped_date`. Deshalb zählen wir Produkte pro Tag über eindeutige URLs statt nur über Zeilen. Fehlende Preise sind bei einzelnen Produkten möglich und werden bei Preisanalysen ausgeschlossen.

In [3]:
display(pd.DataFrame({
    'Datentyp': produkte.dtypes.astype(str),
    'Fehlende Werte': produkte.isna().sum(),
    'Fehlende Werte (%)': (produkte.isna().mean() * 100).round(2),
}))

for column in ['scraped_date', 'produkt_url', 'preis', 'old_price', 'currency', 'suchbegriff']:
    if column not in produkte.columns:
        raise ValueError(f'Erwartete Spalte fehlt: {column}')

produkte['scraped_date'] = pd.to_datetime(produkte['scraped_date'], errors='coerce').dt.normalize()
produkte['preis'] = pd.to_numeric(produkte['preis'], errors='coerce')
produkte['old_price'] = pd.to_numeric(produkte['old_price'], errors='coerce')
produkte['currency'] = produkte['currency'].fillna('EUR').astype(str).str.upper().str.strip()
produkte['produkt_url'] = produkte['produkt_url'].fillna('').astype(str).str.strip()
produkte = produkte.dropna(subset=['scraped_date'])
produkte = produkte[produkte['produkt_url'].ne('')].copy()
produkte = produkte.drop_duplicates(['scraped_date', 'produkt_url'], keep='last')

print(f'Analyse-Zeilen nach Bereinigung: {len(produkte):,}')
print(f'Zeilen ohne gültiges Datum oder Produkt-URL wurden ausgeschlossen.')

,Datentyp,Fehlende Werte,Fehlende Werte (%)
id,int64,0,0.00
scraped_date,str,0,0.00
suchbegriff,str,0,0.00
titel,str,0,0.00
marke,str,0,0.00
preis,float64,0,0.00
old_price,float64,3375,47.34
currency,str,0,0.00
bild_url,str,0,0.00
produkt_url,str,0,0.00


Analyse-Zeilen nach Bereinigung: 7,129
Zeilen ohne gültiges Datum oder Produkt-URL wurden ausgeschlossen.


## 3. Überblick: wie lange wurde gescraped?

Wir unterscheiden **Scrape-Tage mit mindestens einem Produkt** von **Kalendertagen im betrachteten Zeitraum**. Ein Tag ohne gespeicherte Zeile wird im Kalender mit null Produkten aufgeführt. Das zeigt eine Datenlücke, aber allein noch nicht, ob der Workflow gar nicht lief oder nur nichts gespeichert hat.

In [4]:
daily = (
    produkte.groupby('scraped_date')
    .agg(
        produktzeilen=('produkt_url', 'size'),
        produkte=('produkt_url', 'nunique'),
        suchbegriffe=('suchbegriff', 'nunique'),
        preise_vorhanden=('preis', 'count'),
    )
    .sort_index()
)
calendar = pd.date_range(daily.index.min(), daily.index.max(), freq='D', name='scraped_date')
daily = daily.reindex(calendar, fill_value=0)
daily.index.name = 'scraped_date'
daily['scrape_vorhanden'] = daily['produkte'].gt(0)
daily['luecke'] = ~daily['scrape_vorhanden']
lueckentage = daily.index[daily['luecke']]

print(f"Scrape-Tage mit Daten: {int(daily['scrape_vorhanden'].sum())}")
print(f'Kalendertage im Zeitraum: {len(daily)}')
print(f'Zeitraum: {daily.index.min():%d.%m.%Y} bis {daily.index.max():%d.%m.%Y}')
print(f'Tage ohne gespeicherte Produkte: {len(lueckentage)}')
display(daily.reset_index().assign(scraped_date=lambda x: x['scraped_date'].dt.strftime('%Y-%m-%d')))

Scrape-Tage mit Daten: 14
Kalendertage im Zeitraum: 14
Zeitraum: 18.09.2026 bis 01.10.2026
Tage ohne gespeicherte Produkte: 0


,scraped_date,produktzeilen,produkte,suchbegriffe,preise_vorhanden,scrape_vorhanden,luecke
0,2026-09-18,566,566,103,566,True,False
1,2026-09-19,509,509,103,509,True,False
2,2026-09-20,506,506,103,506,True,False
3,2026-09-21,501,501,103,501,True,False
4,2026-09-22,504,504,102,504,True,False
5,2026-09-23,506,506,103,506,True,False
6,2026-09-24,506,506,103,506,True,False
7,2026-09-25,501,501,103,501,True,False
8,2026-09-26,508,508,103,508,True,False
9,2026-09-27,504,504,103,504,True,False


### Diagramm 1: gespeicherte Produkte pro Tag

Die Balken zeigen die Zahl eindeutiger Produkte pro Tag. Eine gestrichelte Linie markiert den Median als robuste Vergleichsbasis. Fehlende Tage stehen ausdrücklich als Null-Balken im Diagramm. Kleine Schwankungen sind nicht automatisch Fehler: Suchergebnisse und Produktverfügbarkeit können sich ändern.

In [5]:
daily_chart_data = daily.reset_index()
median_products = float(daily.loc[daily['scrape_vorhanden'], 'produkte'].median())
base = alt.Chart(daily_chart_data).encode(
    x=alt.X('scraped_date:T', title='Scrape-Datum', axis=alt.Axis(format='%d.%m.%Y', labelAngle=-35)),
)
bars = base.mark_bar().encode(
    y=alt.Y('produkte:Q', title='Eindeutige Produkte'),
    color=alt.condition('datum.luecke', alt.value('#d62728'), alt.value('#4c78a8')),
    tooltip=[alt.Tooltip('scraped_date:T', title='Datum'), 'produkte:Q', 'suchbegriffe:Q', 'luecke:N'],
)
median = alt.Chart(pd.DataFrame({'median': [median_products]})).mark_rule(color='#f58518', strokeDash=[6, 4]).encode(y='median:Q')
(bars + median).properties(width=780, height=320, title='Gespeicherte Produkte je Kalendertag').interactive()

alt.LayerChart(...)

### Diagramm 2: Anzahl beobachteter Suchbegriffe

Der Scraper sucht pro Tag nach vielen Begriffen. Die Tabelle zählt nur Suchbegriffe, die auch in gespeicherten Produktzeilen vorkommen. Begriffe ohne Treffer und fehlgeschlagene Anfragen erscheinen nicht in der Datenbank. Die Zahl ist daher ein **Hinweis auf Abdeckung**, aber kein verlässlicher Erfolgszähler pro Anfrage.

In [6]:
terms_chart = alt.Chart(daily_chart_data).mark_line(point=True, color='#72b7b2').encode(
    x=alt.X('scraped_date:T', title='Scrape-Datum', axis=alt.Axis(format='%d.%m.%Y', labelAngle=-35)),
    y=alt.Y('suchbegriffe:Q', title='Suchbegriffe mit gespeicherten Treffern'),
    tooltip=[alt.Tooltip('scraped_date:T', title='Datum'), 'suchbegriffe:Q', 'produkte:Q'],
).properties(width=780, height=260, title='Beobachtete Suchbegriffe pro Tag').interactive()
terms_chart

alt.Chart(...)

## 4. Produkte an zwei aufeinanderfolgenden Tagen vergleichen

Die Kennzahl **„vom Vortag wiedergefunden (%)“** beantwortet: Welcher Anteil der Produkte von gestern ist heute noch vorhanden? Zusätzlich berechnen wir den **Jaccard-Ähnlichkeitswert**: gemeinsame Produkte geteilt durch alle Produkte, die an einem der beiden Tage vorkamen. Beide Kennzahlen liegen zwischen 0 % (keine Überschneidung) und 100 % (identische Mengen).

Verglichen werden nur direkt aufeinanderfolgende Kalendertage. Nach einer Datenlücke gibt es keine Vortags-Kennzahl, weil ein Vergleich mit dem letzten Scrape mehrere Tage überspringen würde. Die Werte messen Bestandsüberschneidung, nicht ob ein Scrape-Lauf erfolgreich war.

In [7]:
products_by_day = {
    day: set(group['produkt_url'])
    for day, group in produkte.groupby('scraped_date')
}
comparison_rows = []
for day in daily.index:
    previous_day = day - pd.Timedelta(days=1)
    previous = products_by_day.get(previous_day)
    current = products_by_day.get(day)
    if previous is None or current is None:
        comparison_rows.append({
            'scraped_date': day, 'vortag_produkte': len(previous) if previous is not None else 0,
            'heute_produkte': len(current) if current is not None else 0,
            'gemeinsam': None, 'vom_vortag_wiedergefunden_pct': None,
            'gemeinsam_anteil_heute_pct': None, 'jaccard_pct': None,
        })
        continue
    common = len(previous & current)
    comparison_rows.append({
        'scraped_date': day, 'vortag_produkte': len(previous), 'heute_produkte': len(current),
        'gemeinsam': common,
        'vom_vortag_wiedergefunden_pct': 100 * common / len(previous) if previous else None,
        'gemeinsam_anteil_heute_pct': 100 * common / len(current) if current else None,
        'jaccard_pct': 100 * common / len(previous | current) if previous | current else None,
    })
comparison = pd.DataFrame(comparison_rows)
valid_comparisons = comparison.dropna(subset=['vom_vortag_wiedergefunden_pct'])

if valid_comparisons.empty:
    print('Es gibt keine zwei direkt aufeinanderfolgenden Scrape-Tage für einen Vergleich.')
else:
    print(f"Durchschnittlich wiedergefundener Anteil vom Vortag: {valid_comparisons['vom_vortag_wiedergefunden_pct'].mean():.1f}%")
    print(f"Durchschnittlicher Jaccard-Index: {valid_comparisons['jaccard_pct'].mean():.1f}%")
display(comparison.assign(scraped_date=comparison['scraped_date'].dt.strftime('%Y-%m-%d')).round(1))

Durchschnittlich wiedergefundener Anteil vom Vortag: 69.0%
Durchschnittlicher Jaccard-Index: 53.8%


,scraped_date,vortag_produkte,heute_produkte,gemeinsam,vom_vortag_wiedergefunden_pct,gemeinsam_anteil_heute_pct,jaccard_pct
0,2026-09-18,0,566,NaN,NaN,NaN,NaN
1,2026-09-19,566,509,451.0,79.7,88.6,72.3
2,2026-09-20,509,506,429.0,84.3,84.8,73.2
3,2026-09-21,506,501,412.0,81.4,82.2,69.2
4,2026-09-22,501,504,380.0,75.8,75.4,60.8
5,2026-09-23,504,506,309.0,61.3,61.1,44.1
6,2026-09-24,506,506,333.0,65.8,65.8,49.0
7,2026-09-25,506,501,334.0,66.0,66.7,49.6
8,2026-09-26,501,508,323.0,64.5,63.6,47.1
9,2026-09-27,508,504,335.0,65.9,66.5,49.5


In [8]:
if not valid_comparisons.empty:
    comparison_long = valid_comparisons.melt(
        id_vars='scraped_date',
        value_vars=['vom_vortag_wiedergefunden_pct', 'jaccard_pct'],
        var_name='Kennzahl', value_name='Prozent',
    )
    comparison_chart = alt.Chart(comparison_long).mark_line(point=True).encode(
        x=alt.X('scraped_date:T', title='Tag (Vergleich mit Vortag)', axis=alt.Axis(format='%d.%m.%Y', labelAngle=-35)),
        y=alt.Y('Prozent:Q', title='Überschneidung (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('Kennzahl:N', title='Kennzahl'),
        tooltip=[alt.Tooltip('scraped_date:T', title='Datum'), 'Kennzahl:N', alt.Tooltip('Prozent:Q', format='.1f')],
    ).properties(width=780, height=300, title='Tägliche Produktüberschneidung')
    display(comparison_chart.interactive())
else:
    print('Für dieses Diagramm werden mindestens zwei aufeinanderfolgende Scrape-Tage benötigt.')

alt.Chart(...)

## 5. Wie oft kommen dieselben Produkte vor?

Wir zählen für jede URL, an wie vielen **beobachteten Scrape-Tagen** sie vorkam. „An jedem Tag“ bedeutet hier an jedem Tag mit Daten, nicht an jedem Kalendertag, falls es Lücken gibt. Einzelne Produkte können aus dem Sortiment verschwinden oder wechselnde Suchergebnisse haben. Häufigkeit allein ist deshalb kein Qualitätsurteil über den Scraper.

In [9]:
observed_days = int(daily['scrape_vorhanden'].sum())
frequency = (
    produkte.groupby('produkt_url')
    .agg(tage_gefunden=('scraped_date', 'nunique'), titel=('titel', 'last'), marke=('marke', 'last'))
)
frequency['anteil_beobachteter_tage_pct'] = 100 * frequency['tage_gefunden'] / observed_days
frequency_distribution = (
    frequency.groupby('tage_gefunden').size().rename('anzahl_produkte').reset_index()
)
always_products = frequency[frequency['tage_gefunden'].eq(observed_days)].sort_values('titel')
once_products = frequency[frequency['tage_gefunden'].eq(1)].sort_values('titel')

print(f'Produkte an jedem der {observed_days} beobachteten Scrape-Tage: {len(always_products):,}')
print(f'Produkte genau an einem beobachteten Scrape-Tag: {len(once_products):,}')
print(f"Produkte insgesamt (eindeutige URLs): {len(frequency):,}")
display(frequency_distribution)
display(Markdown('### Beispiel: Produkte, die an jedem Scrape-Tag vorkamen'))
display(always_products.head(30))
display(Markdown('### Beispiel: Produkte, die genau einmal vorkamen'))
display(once_products.head(30))

Produkte an jedem der 14 beobachteten Scrape-Tage: 123
Produkte genau an einem beobachteten Scrape-Tag: 351
Produkte insgesamt (eindeutige URLs): 1,375


,tage_gefunden,anzahl_produkte
0,1,351
1,2,189
2,3,129
3,4,100
4,5,115
5,6,68
6,7,44
7,8,54
8,9,57
9,10,32


### Beispiel: Produkte, die an jedem Scrape-Tag vorkamen

,tage_gefunden,titel,marke,anteil_beobachteter_tage_pct
produkt_url,,,,
https://www.otto.de/p/ansmann-ag-laserpointer-ansmann-laserpointer-2in1-CS08GG0K4/,14,ANSMANN AG Laserpointer Ansmann Laserpointer 2in1,ANSMANN AG,100.0
https://www.otto.de/p/anysun-led-sternenhimmel-sternenhimmel-projektor-mit-musikplayer-nachtlicht-aurora-effekt-led-nachtlicht-polarlicht-mit-fernbedienung-weisse-rauschen-timer-und-sternen-fuer-kinder-erwachsene-deko-S0QDR0PB/,14,ANYSUN LED-Sternenhimmel Sternenhimmel Projektor mit Musikplayer Nachtlicht Aurora Effekt,ANYSUN,100.0
https://www.otto.de/p/adler-donut-maker-ad-3075-7-stueck-1000w-mit-antihaftplatten-kontrollleuchten-1000-w-kompaktes-design-cool-touch-gehaeuse-schnelle-aufheizzeit-CS0JFP0KN/,14,Adler Donut-Maker AD 3075 7 Stück - 1000W mit Antihaftplatten & Kontrollleuchten,Adler,100.0
https://www.otto.de/p/altersocks-freizeitsocken-lustige-socken-sushi-socken-damen-herren-unisex-groesse-36-45-1-paar-S017H0UW/,14,AlterSocks Freizeitsocken Lustige Socken Sushi Socken Damen & Herren Unisex Größe 36 – 45 (1 Paar),AlterSocks,100.0
https://www.otto.de/p/apollo-longboard-twin-tip-dt-longboard-39-aus-holz-mehrlagig-verleimt-fuer-idealen-flex-stabilitaet-S002X00G/,14,"Apollo Longboard Twin Tip DT Longboard 39""",Apollo,100.0
https://www.otto.de/p/apollo-skateboard-kinderskateboard-28-kinder-longboard-fuer-kids-und-teens-abec-3-und-aluminium-achsen-S002X00Q/,14,"Apollo Skateboard Kinderskateboard 28"" Kinder",Apollo,100.0
https://www.otto.de/p/athlix-3d-drucker-3d-drucker-mehrfarbig-kompatibel-mit-pla-tpu-1-75-mm-filamenten-100-100-100-mm-bauraum-fdm-drucktechnologie-S00G403E/,14,"Athlix 3D-Drucker 3D Drucker mehrfarbig kompatibel mit PLA, TPU, 1,75 mm Filamenten",Athlix,100.0
https://www.otto.de/p/bestway-badespielzeug-bestway-riesen-flamingo-schwimmtier-191x178-cm-zum-1-tlg-S062X06S/,14,BESTWAY Badespielzeug Bestway riesen Flamingo Schwimmtier 191x178 cm zum,BESTWAY,100.0
https://www.otto.de/p/brubaker-kuscheltier-pluesch-einhorn-21-cm-1-st-stofftier-plueschtier-pluescheinhorn-S0R1G0CP/,14,BRUBAKER Kuscheltier Plüsch Einhorn 21 cm,BRUBAKER,100.0


### Beispiel: Produkte, die genau einmal vorkamen

,tage_gefunden,titel,marke,anteil_beobachteter_tage_pct
produkt_url,,,,
https://www.otto.de/p/100years-suessungsmittel-100years-premium-kandissticks-braun-S0TJK09U/,1,100years Süßungsmittel 100years - Premium Kandissticks braun,100years,7.142857
https://www.otto.de/p/acgam-buerostuhl-ergonomisch-schreibtischstuhl-buero-stuhl-gaming-stuhl-150-kg-unabhaengige-rueckenlehnenverstellung-sitztiefenverstellung-3d-armlehnen-eine-2d-verstellbare-kopfstuetze-und-ein-innovativer-gewichtsabhaengiger-kippmechanismus-smartweight-technologie-3d-adaptive-lendenwirbelstuetze-CS0NJ50E3/,1,"ACGAM Bürostuhl Ergonomisch Schreibtischstuhl, Büro Stuhl & Gaming Stuhl, 150 kg",ACGAM,7.142857
https://www.otto.de/p/afksmile-dekokissen-weiches-oktopus-kuscheltier-realistisches-cuddly-octopus-plueschtier-S0PJF0ST/,1,"AFKSMILE Dekokissen Weiches Oktopus Kuscheltier, realistisches Cuddly Octopus Plüschtier",AFKSMILE,7.142857
https://www.otto.de/p/aibistar-retro-cd-player-mit-bluetooth-radio-usb-stereo-5m-fernbedienung-cd-player-fm-usb-aux-plattenspieler-design-nostalgischer-charme-S08K100P/,1,"AIBISTAR Retro CD-Player mit Bluetooth, Radio & USB – Stereo, 5m Fernbedienung CD-Player",AIBISTAR,7.142857
https://www.otto.de/p/airrobo-nass-trocken-saugroboter-akku-staubsauger-staubsauger-roboter-mit-wischfunktion-55-w-laser-kartierung-laser-hinderniserkennung-automatische-entleerung-des-staubbehalters-alle-60-tage-13000pa-antitangle-gummibuerste-ideal-fuer-tierhaare-240-min-laufzeit-S0XGP09O/,1,"AIRROBO Nass-Trocken-Saugroboter akku staubsauger,Staubsauger Roboter mit Wischfunktion",AIRROBO,7.142857
https://www.otto.de/p/ams-funkwanduhr-exklusiv-modell-quarzuhr-made-in-germany-esszimmer-wohnzimmer-buero-1792759657/,1,AMS Funkwanduhr,AMS,7.142857
https://www.otto.de/p/anycubic-3d-drucker-anycubic-kobra-s1-combo-3d-drucker-mehrfarbig-CS0CDH03L/,1,ANYCUBIC 3D-Drucker Anycubic Kobra S1 Combo 3D Drucker mehrfarbig,ANYCUBIC,7.142857
https://www.otto.de/p/alecto-discolicht-le-180-stroboskop-einstellbare-farbeffekte-einstellbare-lichteffekte-sound-to-light-empfindlichkeitseinstellung-dmx-steuerung-led-8-lichteffekte-stroboskop-soundgesteuert-dmx-ein-ausgang-S086F0M9/,1,Alecto Discolicht LE-180,Alecto,7.142857
https://www.otto.de/p/anker-nebula-capsule3-mini-beamer-S03J5009/,1,Anker Nebula Capsule3 Mini Beamer,Anker,7.142857


In [10]:
frequency_chart = alt.Chart(frequency_distribution).mark_bar(color='#54a24b').encode(
    x=alt.X('tage_gefunden:O', title='Anzahl Scrape-Tage mit Produkt'),
    y=alt.Y('anzahl_produkte:Q', title='Anzahl unterschiedlicher Produkte'),
    tooltip=['tage_gefunden:O', 'anzahl_produkte:Q'],
).properties(width=650, height=300, title='Verteilung der Produkt-Häufigkeit')
frequency_chart

alt.Chart(...)

## 6. Produktbestand als Heatmap

Jede Zelle zeigt, ob ein Produkt an einem Tag gefunden wurde. Die bisherige Auswahl der häufigsten Produkte bestand fast nur aus Artikeln, die an allen Tagen vorkamen – dadurch war die Heatmap fast komplett blau. Jetzt wählen wir bewusst Produkte aus mehreren Häufigkeitsgruppen, damit Unterschiede sichtbar sind. `TOP_N` steuert die ungefähre Zahl der Beispielprodukte. Fahre über eine Zelle, um den vollständigen Produkttitel zu sehen.

In [11]:
TOP_N = 20  # Mitgestalten: Anzahl der Produkte in der Heatmap
def frequency_group(days):
    if days == observed_days:
        return 'Jeden Scrape-Tag'
    if days == 1:
        return 'Genau 1 Tag'
    if days <= 4:
        return '2–4 Tage'
    if days <= 9:
        return '5–9 Tage'
    return '10+ Tage'

group_order = ['Genau 1 Tag', '2–4 Tage', '5–9 Tage', '10+ Tage', 'Jeden Scrape-Tag']
frequency['gruppe'] = frequency['tage_gefunden'].map(frequency_group)
available_groups = [group for group in group_order if frequency['gruppe'].eq(group).any()]
samples_per_group = max(1, TOP_N // len(available_groups))
selected_urls = []
for group in available_groups:
    candidates = frequency[frequency['gruppe'].eq(group)]
    sample_size = min(samples_per_group, len(candidates))
    selected_urls.extend(candidates.sample(n=sample_size, random_state=42).index.tolist())
if len(selected_urls) < TOP_N:
    remaining = frequency.drop(index=selected_urls)
    extra = remaining.sample(n=min(TOP_N - len(selected_urls), len(remaining)), random_state=42)
    selected_urls.extend(extra.index.tolist())
selected_frequency = frequency.loc[selected_urls].copy()
selected_frequency['gruppe'] = selected_frequency['tage_gefunden'].map(frequency_group)
selected_frequency['sort_gruppe'] = selected_frequency['gruppe'].map({group: i for i, group in enumerate(group_order)})
selected_frequency = selected_frequency.sort_values(['sort_gruppe', 'tage_gefunden', 'titel'])
selected_urls = selected_frequency.index.tolist()

presence = produkte[produkte['produkt_url'].isin(selected_urls)].pivot_table(
    index='produkt_url', columns='scraped_date', values='titel',
    aggfunc='size', fill_value=0,
).reindex(columns=daily.index, fill_value=0)
presence_long = (
    presence.rename_axis('produkt_url', axis='index').reset_index()
    .melt(id_vars='produkt_url', var_name='Datum', value_name='Gefunden')
)
presence_long['Datum'] = pd.to_datetime(presence_long['Datum']).dt.strftime('%d.%m.')
presence_long['Vollständiger Titel'] = presence_long['produkt_url'].map(frequency['titel'])
presence_long['Scrape-Tage'] = presence_long['produkt_url'].map(frequency['tage_gefunden'])
presence_long['Gruppe'] = presence_long['produkt_url'].map(frequency['gruppe'])
presence_long['Produkt'] = presence_long.apply(
    lambda row: f"{row['Scrape-Tage']}/{observed_days} Tage | {row['Vollständiger Titel'][:42]}",
    axis=1,
)
presence_long['Status'] = presence_long['Gefunden'].map({0: 'Nicht gefunden', 1: 'Gefunden'})
date_labels = [day.strftime('%d.%m.') for day in daily.index]
product_order = selected_frequency.apply(
    lambda row: f"{row['tage_gefunden']}/{observed_days} Tage | {row['titel'][:42]}", axis=1
).tolist()
presence_chart = alt.Chart(presence_long).mark_rect().encode(
    x=alt.X('Datum:O', title='Scrape-Datum', sort=date_labels, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Produkt:N', title='Produkt | gefundene Scrape-Tage', sort=product_order, axis=alt.Axis(labelLimit=340)),
    color=alt.Color('Status:N', title='Tagesergebnis', scale=alt.Scale(domain=['Gefunden', 'Nicht gefunden'], range=['#2367a5', '#e8edf2'])),
    tooltip=['Vollständiger Titel:N', 'Gruppe:N', alt.Tooltip('Datum:O', title='Datum'), 'Status:N'],
).properties(width=850, height=max(300, 25 * len(presence.index)), title=f'Produktverlauf: {len(presence.index)} Beispiele aus verschiedenen Häufigkeitsgruppen')
presence_chart

alt.Chart(...)

## 7. Preisverteilung und Preisbereiche

Preise sind nur sinnvoll vergleichbar, wenn dieselbe Währung verwendet wird. Die Preisbereiche unten betrachten die **Produkte des neuesten Scrape-Tages**, damit ein täglich wiedergefundenes Produkt nicht mehrfach gezählt wird. Das folgende Zeitdiagramm verwendet dagegen die gültigen EUR-Preise **aller Scrape-Tage**. Sehr extreme Preise werden nur in diesem Diagramm oberhalb des 99. Perzentils ausgeblendet, damit die typischen Artikelpreise lesbar bleiben; sie bleiben in der Datenbank und den anderen Auswertungen enthalten.

In [12]:
price_day = produkte['scraped_date'].max()
latest = produkte[produkte['scraped_date'].eq(price_day)].copy()
display(latest['currency'].value_counts(dropna=False).rename_axis('Währung').to_frame('Produkte am neuesten Scrape-Tag'))
eur = latest[
    latest['currency'].eq('EUR') & latest['preis'].notna() & latest['preis'].gt(0)
].copy()
PRICE_BINS = [0, 10, 25, 50, 100, 250, 500, 1000, float('inf')]  # Mitgestalten: Preisgrenzen in Euro
PRICE_LABELS = ['0–10 €', '10–25 €', '25–50 €', '50–100 €', '100–250 €', '250–500 €', '500–1.000 €', 'über 1.000 €']
eur['preisbereich'] = pd.cut(eur['preis'], bins=PRICE_BINS, labels=PRICE_LABELS, right=False)
price_distribution = (
    eur['preisbereich'].value_counts(sort=False).rename('produkte').to_frame()
    .assign(prozent=lambda x: 100 * x['produkte'] / max(len(eur), 1))
    .reset_index(names='preisbereich')
)
print(f'Produkte mit gültigem EUR-Preis am {price_day:%d.%m.%Y}: {len(eur):,}')
if not eur.empty:
    print(f"Median: {eur['preis'].median():.2f} € | Mittelwert: {eur['preis'].mean():.2f} € | 90. Perzentil: {eur['preis'].quantile(.9):.2f} €")
display(price_distribution.round(2))

,Produkte am neuesten Scrape-Tag
Währung,
EUR,509


Produkte mit gültigem EUR-Preis am 01.10.2026: 509
Median: 49.95 € | Mittelwert: 1157.47 € | 90. Perzentil: 341.00 €


,preisbereich,produkte,prozent
0,0–10 €,32,6.29
1,10–25 €,102,20.04
2,25–50 €,127,24.95
3,50–100 €,101,19.84
4,100–250 €,76,14.93
5,250–500 €,40,7.86
6,500–1.000 €,17,3.34
7,über 1.000 €,14,2.75


In [13]:
price_chart = alt.Chart(price_distribution).mark_bar(color='#e45756').encode(
    x=alt.X('preisbereich:N', title='Preisbereich (EUR)', sort=PRICE_LABELS, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('produkte:Q', title='Produktbeobachtungen'),
    tooltip=['preisbereich:N', 'produkte:Q', alt.Tooltip('prozent:Q', format='.1f', title='Anteil (%)')],
).properties(width=700, height=320, title=f'Artikel nach Preisbereich am {price_day:%d.%m.%Y}')
price_chart

alt.Chart(...)

### Preisverteilung über alle Scrape-Tage

Jede Box zeigt die Preise des jeweiligen Tages. Die y-Achse ist logarithmisch, damit günstige und teure Artikel zugleich sichtbar sind. Zur besseren Lesbarkeit begrenzen wir die Darstellung auf das 99. Perzentil; die ausgeblendeten Extremwerte werden gezählt und nicht aus den Daten gelöscht. So lässt sich erkennen, ob sich der typische Preisbestand über die Tage verändert.

In [14]:
price_history = produkte[
    produkte['currency'].eq('EUR') & produkte['preis'].notna() & produkte['preis'].gt(0)
].copy()
if not price_history.empty:
    price_cap = float(price_history['preis'].quantile(0.99))
    price_history_plot = price_history[price_history['preis'].le(price_cap)].copy()
    price_history_plot['Datum'] = price_history_plot['scraped_date'].dt.strftime('%d.%m.')
    date_labels = [day.strftime('%d.%m.') for day in daily.index]
    hidden_extremes = len(price_history) - len(price_history_plot)
    print(f"Im Diagramm: {len(price_history_plot):,} von {len(price_history):,} Preisbeobachtungen. {hidden_extremes} Preise über {price_cap:,.2f} € (99. Perzentil) sind nur aus der Grafik ausgeblendet.")
    boxplot = alt.Chart(price_history_plot).mark_boxplot(extent=1.5).encode(
        x=alt.X('Datum:O', title='Scrape-Datum', sort=date_labels, axis=alt.Axis(labelAngle=-35)),
        y=alt.Y('preis:Q', title='Preis (EUR, logarithmisch)', scale=alt.Scale(type='log', domain=[max(0.01, float(price_history_plot['preis'].min())), price_cap])),
        tooltip=[alt.Tooltip('Datum:O', title='Scrape-Datum'), alt.Tooltip('count(preis):Q', title='Produkte an diesem Tag')],
    ).properties(width=850, height=380, title='Tägliche Preisverteilung über alle Scrape-Tage (oberste 1 % ausgeblendet)')
    display(boxplot)
else:
    print('Keine positiven EUR-Preise für ein Preisdiagramm vorhanden.')

Im Diagramm: 7,060 von 7,129 Preisbeobachtungen. 69 Preise über 2,890.00 € (99. Perzentil) sind nur aus der Grafik ausgeblendet.


alt.Chart(...)

## 8. Auffällige Preis- und Datenqualitätssignale

Dieser Abschnitt zeigt einfache Prüfhinweise, keine automatische Fehlerdiagnose. Stark abweichende Tagesmengen, fehlende Preise oder große Preisänderungen können eine nähere Untersuchung lohnen. Ein Produktpreis kann sich tatsächlich ändern; deshalb nennen wir Abweichungen **Prüfkandidaten**, nicht Fehler.

In [15]:
quality_summary = pd.Series({
    'Produktbeobachtungen ohne Preis': int(produkte['preis'].isna().sum()),
    'Anteil ohne Preis (%)': round(100 * produkte['preis'].isna().mean(), 2),
    'Produktbeobachtungen ohne Suchbegriff': int(produkte['suchbegriff'].isna().sum()),
    'Produktbeobachtungen ohne Titel': int(produkte['titel'].isna().sum()) if 'titel' in produkte else None,
    'Eindeutige Produkte': int(produkte['produkt_url'].nunique()),
    'Währungen': int(produkte['currency'].nunique()),
}, name='Wert').to_frame()
display(quality_summary)

daily['veraenderung_prozent'] = daily['produkte'].pct_change() * 100
daily['abweichung_vom_median'] = daily['produkte'] - median_products
display(daily[['produkte', 'suchbegriffe', 'veraenderung_prozent', 'abweichung_vom_median']].round(1))

,Wert
Produktbeobachtungen ohne Preis,0.0
Anteil ohne Preis (%),0.0
Produktbeobachtungen ohne Suchbegriff,0.0
Produktbeobachtungen ohne Titel,0.0
Eindeutige Produkte,1375.0
Währungen,1.0


,produkte,suchbegriffe,veraenderung_prozent,abweichung_vom_median
scraped_date,,,,
2026-09-18,566,103,NaN,60.0
2026-09-19,509,103,-10.1,3.0
2026-09-20,506,103,-0.6,0.0
2026-09-21,501,103,-1.0,-5.0
2026-09-22,504,102,0.6,-2.0
2026-09-23,506,103,0.4,0.0
2026-09-24,506,103,0.0,0.0
2026-09-25,501,103,-1.0,-5.0
2026-09-26,508,103,1.4,2.0


## 9. Was sagen diese Daten über den Scraper aus?

### Was wir messen können
- Wie viele Tage tatsächlich gespeicherte Produktdaten enthalten und welche Kalendertage dazwischen ohne Daten liegen.
- Wie viele eindeutige Produkte und wie viele Suchbegriffe mit Treffern pro Tag gespeichert wurden.
- Wie viele Produkte über mehrere Tage wiederkehren und wie stark sich aufeinanderfolgende Tagesbestände überschneiden.
- Wie sich gültige EUR-Preise über frei anpassbare Preisbereiche verteilen.

### Was die Datenbank allein nicht beweisen kann
- Ein fehlender Tag bedeutet **keine Zeilen**, aber verrät nicht, ob der GitHub-Workflow ausgelassen wurde, die Website nicht antwortete oder ein Lauf ohne speicherbare Treffer endete.
- Der Scraper speichert fehlgeschlagene Begriffe nur in der Laufzeit-Ausgabe. `suchbegriff` steht nur bei gespeicherten Produkten. Außerdem ersetzt `INSERT OR REPLACE` bei gleicher URL am selben Tag den Datensatz und damit gegebenenfalls den vorherigen Suchbegriff.
- Ein ähnlicher Tagesumfang bedeutet nicht zwingend, dass jeder Suchbegriff erfolgreich war; ein fehlgeschlagener Begriff kann durch zusätzliche Treffer anderer Begriffe verdeckt werden.

### Sinnvolle nächste Verbesserung für zuverlässiges Monitoring
Speichere jeden Lauf und jeden Suchbegriff zusätzlich in Protokolltabellen, z. B. `scrape_runs` (Start, Ende, Status, gespeicherte Gesamtzahl) und `scrape_term_runs` (Suchbegriff, Status, Trefferzahl, Fehlertext). Dann lassen sich **Workflow-Ausfall**, **Suchbegriff-Fehler** und **keine Treffer** getrennt erkennen. Verknüpfe diese Informationen idealerweise auch als GitHub Actions Zusammenfassung. Die aktuellen Datenbankanalysen bleiben nützlich, messen aber den resultierenden Produktbestand und nicht direkt den technischen Erfolg einzelner Requests.

## 10. Kleine Übung: eine eigene Frage stellen

1. Ändere `TOP_N` in Abschnitt 6 und beobachte, wie die Heatmap lesbarer oder dichter wird.
2. Passe `PRICE_BINS` und `PRICE_LABELS` in Abschnitt 7 an deine interessanten Preisklassen an. Die Anzahl der Grenzen muss immer genau eins größer sein als die Anzahl der Labels.
3. Sortiere `frequency` nach `tage_gefunden`, um besonders häufige Produkte zu untersuchen, oder filtere `produkte` nach einem `suchbegriff`.
4. Wähle danach **Kernel → Restart & Run All**, damit alle Ergebnisse mit deinen Einstellungen frisch berechnet werden.

Damit hast du eine reproduzierbare Bestandsanalyse und kannst Kennzahlen gezielt verändern, statt nur fertige Diagramme anzusehen.